# Quickstart

Run one typed policy interaction with TDHook activation caching.

In [ ]:
import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from tdhook.latent import ActivationCaching
from tdhook.workflow import Workflow
from xdrl import (
    BatchSemantics,
    InteractionContract,
    InteractionPhase,
    KeyPresence,
    KeyRole,
    KeySchema,
    ModelRole,
    RuntimeInteractionContext,
    TDHookWorkflowRunner,
    TensorDictSchema,
)

inputs = TensorDictSchema(
    (KeySchema("observation", KeyRole.OBSERVATION, KeyPresence.REQUIRED),), BatchSemantics(("env",))
)
outputs = TensorDictSchema((KeySchema("action", KeyRole.ACTION, KeyPresence.PRODUCED),), BatchSemantics(("env",)))
policy = TensorDictModule(torch.nn.Linear(4, 2), in_keys=["observation"], out_keys=["action"])
batch = TensorDict({"observation": torch.randn(8, 4)}, batch_size=[8])
contract = InteractionContract(
    "policy:evaluation:0",
    ModelRole.ACTOR,
    InteractionPhase.EVALUATION,
    "policy",
    inputs,
    outputs,
    module_training=False,
)
interaction = RuntimeInteractionContext(contract, policy, batch)
workflow = Workflow(ActivationCaching("module", cache_key=("activations", "head")))
execution = TDHookWorkflowRunner(interaction).run(workflow, batch.clone(), code_revision="example-revision")
assert execution.data["action"].shape == (8, 2)
assert "module" in execution.data["activations", "head"]
execution.provenance.model_calls